# 02 — K-Means Clustering Model

**Nguồn dữ liệu**: MySQL → bảng `cleaned_cars`

**Mục tiêu**:
1. Lấy dữ liệu sạch từ DB
2. Scale features bằng `RobustScaler`
3. Xác định K tối ưu (Elbow Method + Silhouette Score)
4. Train K-Means và phân tích từng cluster
5. Lưu nhãn cluster ngược về DB (cột `cluster_label`)

> ⚠️ **Yêu cầu**: Đã chạy `python -m src.etl_pipeline` trước.

## 0. Import & Cấu hình

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Thư mục lưu hình ảnh output
OUTPUT_DIR = ROOT / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
import matplotlib.cm as cm

from src.db_connector import DBConnector

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 5)})

RANDOM_STATE = 42
print(f'✓ Import thành công')
print(f'✓ Output dir: {OUTPUT_DIR}')

## 1. Load dữ liệu từ MySQL

In [ ]:
db = DBConnector()
conn = db.get_connection()

df = pd.read_sql("""
    SELECT 
        id, ten_xe, gia, nam_sx, km_da_di,
        so_ghe, nhien_lieu, kieu_dang, tinh_trang,
        hop_so, xuat_xu, log_gia, log_km
    FROM cleaned_cars
    ORDER BY id
""", conn)

print(f'Dữ liệu từ DB: {df.shape[0]:,} records, {df.shape[1]} cột')
df.head()

## 2. Chuẩn bị Features cho Clustering

In [ ]:
# Features dùng để clustering
FEATURES = ['log_gia', 'log_km', 'nam_sx']

X = df[FEATURES].dropna()
valid_idx = X.index  # Giữ lại index gốc để map lại sau

# Scale bằng RobustScaler (ít nhạy cảm với outlier hơn StandardScaler)
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

print(f'Features: {FEATURES}')
print(f'Shape sau scaling: {X_scaled.shape}')
print(f'\nThống kê sau RobustScaler:')
pd.DataFrame(X_scaled, columns=FEATURES).describe().round(3)

## 3. Tìm K tối ưu — Elbow Method

In [ ]:
k_range = range(2, 11)
inertias = []
sil_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    inertias.append(kmeans.inertia_)
    sil = silhouette_score(X_scaled, labels)
    sil_scores.append(sil)
    print(f'K={k}: Inertia={kmeans.inertia_:,.0f} | Silhouette={sil:.4f}')

print(f'\n→ K tối ưu theo Silhouette: K={k_range.start + np.argmax(sil_scores)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow curve
axes[0].plot(list(k_range), inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_title('Elbow Method — Inertia', fontweight='bold')
axes[0].set_xlabel('Số cụm K')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].grid(True, alpha=0.3)

# Silhouette curve
axes[1].plot(list(k_range), sil_scores, 'rs-', linewidth=2, markersize=8)
best_k_idx = np.argmax(sil_scores)
axes[1].axvline(x=list(k_range)[best_k_idx], color='green', linestyle='--',
                label=f'K={list(k_range)[best_k_idx]} (tốt nhất)')
axes[1].set_title('Silhouette Score theo K', fontweight='bold')
axes[1].set_xlabel('Số cụm K')
axes[1].set_ylabel('Silhouette Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Tìm K tối ưu cho K-Means', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'kmeans_01_elbow_silhouette.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'✓ Đã lưu → output/kmeans_01_elbow_silhouette.png')

## 4. Train K-Means với K tối ưu

In [ ]:
# Chọn K — thay đổi nếu muốn
K_OPTIMAL = 4

kmeans_final = KMeans(n_clusters=K_OPTIMAL, random_state=RANDOM_STATE, n_init=10)
cluster_labels = kmeans_final.fit_predict(X_scaled)

# Gắn nhãn cluster vào DataFrame gốc
df_result = df.loc[valid_idx].copy()
df_result['cluster'] = cluster_labels

print(f'K = {K_OPTIMAL}')
print(f'Silhouette Score cuối: {silhouette_score(X_scaled, cluster_labels):.4f}')
print(f'\nSố lượng mỗi cụm:')
print(df_result['cluster'].value_counts().sort_index())

## 5. Phân tích đặc trưng từng Cluster

In [ ]:
# Thống kê trung bình của từng cụm
cluster_stats = df_result.groupby('cluster').agg(
    so_luong=('id', 'count'),
    gia_tb=('gia', 'mean'),
    gia_min=('gia', 'min'),
    gia_max=('gia', 'max'),
    km_tb=('km_da_di', 'mean'),
    nam_sx_tb=('nam_sx', 'mean'),
).round(0)

# Chuyển giá sang tỷ VND
for col in ['gia_tb', 'gia_min', 'gia_max']:
    cluster_stats[col] = (cluster_stats[col] / 1e9).round(2)

cluster_stats.columns = ['Số lượng', 'Giá TB (tỷ)', 'Giá Min (tỷ)', 'Giá Max (tỷ)', 'Km TB', 'Năm SX TB']
print('=== Thống kê từng Cluster ===')
cluster_stats

## 6. Visualize Clusters

In [ ]:
palette = sns.color_palette('Set1', K_OPTIMAL)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Scatter: Log(Giá) vs Năm SX
for k in range(K_OPTIMAL):
    mask = df_result['cluster'] == k
    axes[0].scatter(
        df_result.loc[mask, 'nam_sx'],
        df_result.loc[mask, 'log_gia'],
        label=f'Cluster {k} (n={mask.sum()})',
        alpha=0.5, s=20, color=palette[k]
    )

axes[0].set_title('Cluster: Log(Giá) vs Năm SX', fontweight='bold')
axes[0].set_xlabel('Năm sản xuất')
axes[0].set_ylabel('Log(Giá + 1)')
axes[0].legend(markerscale=2)

# Scatter: Log(Giá) vs Log(Km)
for k in range(K_OPTIMAL):
    mask = df_result['cluster'] == k
    axes[1].scatter(
        df_result.loc[mask, 'log_km'],
        df_result.loc[mask, 'log_gia'],
        label=f'Cluster {k}',
        alpha=0.5, s=20, color=palette[k]
    )

axes[1].set_title('Cluster: Log(Giá) vs Log(Km)', fontweight='bold')
axes[1].set_xlabel('Log(Km + 1)')
axes[1].set_ylabel('Log(Giá + 1)')
axes[1].legend(markerscale=2)

plt.suptitle(f'K-Means Clustering (K={K_OPTIMAL})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'kmeans_02_scatter_clusters.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'✓ Đã lưu → output/kmeans_02_scatter_clusters.png')

In [ ]:
# Silhouette Plot chi tiết
sil_vals = silhouette_samples(X_scaled, cluster_labels)

fig, ax = plt.subplots(figsize=(9, 6))
y_lower = 10

for k in range(K_OPTIMAL):
    kth_sil = np.sort(sil_vals[cluster_labels == k])
    size_k = kth_sil.shape[0]
    y_upper = y_lower + size_k

    color = cm.nipy_spectral(float(k) / K_OPTIMAL)
    ax.fill_betweenx(np.arange(y_lower, y_upper),
                     0, kth_sil, facecolor=color, edgecolor=color, alpha=0.7)
    ax.text(-0.05, y_lower + 0.5 * size_k, str(k), fontsize=12)
    y_lower = y_upper + 10

avg_sil = silhouette_score(X_scaled, cluster_labels)
ax.axvline(x=avg_sil, color='red', linestyle='--', label=f'Avg = {avg_sil:.3f}')
ax.set_title(f'Silhouette Plot — K={K_OPTIMAL}', fontweight='bold', fontsize=13)
ax.set_xlabel('Silhouette Coefficient')
ax.set_ylabel('Cluster')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'kmeans_03_silhouette_plot.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'✓ Đã lưu → output/kmeans_03_silhouette_plot.png')

In [ ]:
# Phân bố đặc trưng phân loại trong từng cluster
cat_cols = ['kieu_dang', 'nhien_lieu', 'xuat_xu', 'tinh_trang']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    ct = pd.crosstab(df_result['cluster'], df_result[col], normalize='index') * 100
    ct.plot(kind='bar', ax=axes[i], colormap='Set3', edgecolor='white', width=0.7)
    axes[i].set_title(f'Phân bố {col} theo Cluster (%)', fontweight='bold')
    axes[i].set_xlabel('Cluster')
    axes[i].set_ylabel('Tỷ lệ (%)')
    axes[i].tick_params(axis='x', rotation=0)
    axes[i].legend(loc='upper right', fontsize=8)

plt.suptitle('Đặc trưng phân loại trong từng Cluster', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'kmeans_04_cluster_categories.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'✓ Đã lưu → output/kmeans_04_cluster_categories.png')

## 7. Lưu nhãn Cluster về MySQL

In [ ]:
# Cập nhật cluster_label vào bảng cleaned_cars
ids = df_result['id'].tolist()
labels = df_result['cluster'].tolist()

db.update_cluster_labels(ids, labels)
print(f'✓ Đã cập nhật cluster_label cho {len(ids):,} records trong DB')

# Xác nhận
check = pd.read_sql("""
    SELECT cluster_label, COUNT(*) as so_luong
    FROM cleaned_cars
    GROUP BY cluster_label
    ORDER BY cluster_label
""", conn)
print('\nKiểm tra phân phối cluster trong DB:')
check

## 8. Đóng kết nối

In [ ]:
db.close()
print('✓ Đã đóng kết nối DB')
print(f'\n=== TÓM TẮT ===')
print(f'K tối ưu: {K_OPTIMAL}')
print(f'Silhouette Score: {silhouette_score(X_scaled, cluster_labels):.4f}')
print(f'Phân bố cluster:')
for k, cnt in df_result["cluster"].value_counts().sort_index().items():
    print(f'  Cluster {k}: {cnt:,} xe')
print(f'\nTất cả biểu đồ đã lưu tại: {OUTPUT_DIR}')